# DLBCL Donor Pipeline Run

This notebook runs the refactored donor pipeline from configuration through folder-structure modeling, image processing, table building, and result inspection.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "dlbcl_pipeline").exists():
    REPO_ROOT = Path.cwd() / "dlbcl-deep-learning"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/Users/taeeonkong/Desktop/DL Project/dlbcl-deep-learning')

In [2]:
from dlbcl_pipeline.classification.classify_tcells import classify_tcells
from dlbcl_pipeline.config import build_local_pipeline_config
from dlbcl_pipeline.export import build_formatted_channels
from dlbcl_pipeline.measurements.aggregation import build_donor_table
from dlbcl_pipeline.model_folder_structure import build_donor_folder_structure
from dlbcl_pipeline.plotting.intensity_distributions import plot_donor_intensity_distributions
from dlbcl_pipeline.process_donor import process_donor


## Configuration

In [3]:
BASE_PATH = Path("/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241")
PROJECT_DATA_ROOT = REPO_ROOT.parent

# One source of truth for the run selection.
# Use {} or None to process every sample/image.
SAMPLES_TO_PROCESS = {1: [1, 2, 3, 4, 5]}

CLEAN_CELL_LIST_PATH = PROJECT_DATA_ROOT / "clean_cell_list.csv"
EXPORT_CHANNELS = ("actin", "ccr7", "cd45ra")
CLEAN_FORMATTED_OUTPUT = True

ANNOUNCE_FILTERS = True
VERBOSE = False

config = build_local_pipeline_config(
    donor_dir=BASE_PATH,
    samples_to_process=SAMPLES_TO_PROCESS,
)

run_selection = {
    "donor_dir": str(config.donor_dir),
    "samples_to_process": config.samples_to_process,
    "images_to_process": config.images_to_process,
    "clean_cell_list": str(CLEAN_CELL_LIST_PATH),
}

run_selection


{'donor_dir': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241',
 'samples_to_process': {1: [1, 2, 3, 4, 5]},
 'images_to_process': {1: (1, 2, 3, 4, 5)},
 'clean_cell_list': '/Users/taeeonkong/Desktop/DL Project/clean_cell_list.csv'}

## Model Folder Structure

In [4]:
donor_folder_structure = build_donor_folder_structure(
    config,
    announce_filters=ANNOUNCE_FILTERS,
)

donor_folder_structure

Restricting to configured images for sample1: 1, 2, 3, 4, 5


DonorFolderStructure(path=PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241'), full_name='01-03-2026 DLBCL 109241', samples=(SampleFolder(name='sample1', path=PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1'), sample_number=1, images=(ImageFolder(name='1', path=PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1'), image_number=1, channels={'actin': PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/Actin-FITC.tif'), 'cd4': PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/CD4-PerCP.tif'), 'cd45ra_PacBlue': PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/CD45RA-PacBlue.tif'), 'cd19car': PosixPath('/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/CD19CAR-AF647.tif'), 'ccr7': PosixPath('/User

In [5]:
structure_rows = []
for sample in donor_folder_structure.samples:
    for image in sample.images:
        structure_rows.append({
            "sample": sample.name,
            "sample_number": sample.sample_number,
            "image": image.name,
            "image_number": image.image_number,
            "image_path": str(image.path),
            "channels": {key: path.name for key, path in image.channels.items()},
        })

structure_rows

[{'sample': 'sample1',
  'sample_number': 1,
  'image': '1',
  'image_number': 1,
  'image_path': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1',
  'channels': {'actin': 'Actin-FITC.tif',
   'cd4': 'CD4-PerCP.tif',
   'cd45ra_PacBlue': 'CD45RA-PacBlue.tif',
   'cd19car': 'CD19CAR-AF647.tif',
   'ccr7': 'CCR7-AF594.tif'}},
 {'sample': 'sample1',
  'sample_number': 1,
  'image': '2',
  'image_number': 2,
  'image_path': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/2',
  'channels': {'actin': 'Actin-FITC.tif',
   'cd4': 'CD4-PerCP.tif',
   'cd45ra_PacBlue': 'CD45RA-PacBlue.tif',
   'cd19car': 'CD19CAR-AF647.tif',
   'ccr7': 'CCR7-AF594.tif'}},
 {'sample': 'sample1',
  'sample_number': 1,
  'image': '3',
  'image_number': 3,
  'image_path': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/3',
  'channels': {'actin': 'Actin-FITC.tif',
   'cd4': 'CD4-PerCP.tif',
   'cd45ra_PacBlu

## Run Processing

In [ ]:
# sometimes, this cell will fail if the kernel/python process is old. 
# to fix this, restart the kernel

donor_result = process_donor(
    config,
    donor_folder_structure=donor_folder_structure,
    verbose=VERBOSE,
)

{
    "success": donor_result.success,
    "error": donor_result.error,
    "total_images": donor_result.total_images,
    "total_processed": donor_result.total_processed,
    "total_failed": donor_result.total_failed,
}




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	darwin 
python version: 	3.12.11 
torch version:  	2.8.0! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




model_type argument is not used in v4.0.1+. Ignoring this argument...
Resizing is depricated in v4.0.1+


  Saving original image: (1002, 1004), dtype: uint16, unique_values: 13179
✓ Visualization saved: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/cellpose_segmentation_visualization.png
Detecting channel files in: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/1/
Found 6 channel files:
  - Actin-FITC.tif
  - CCR7-AF594.tif
  - CD19CAR-AF647.tif
  - CD4-PerCP.tif
  - CD45RA-PacBlue.tif
  - cellpose_prob_map.tif

Preprocessing channels...
  Processing channel: Actin-FITC (sliding paraboloid, 100px)
    Saved: processed_Actin-FITC.tif
  Processing channel: CCR7-AF594 (sliding paraboloid, 100px)
    Saved: processed_CCR7-AF594.tif
  Processing channel: CD19CAR-AF647 (sliding paraboloid, 100px)
    Saved: processed_CD19CAR-AF647.tif
  Processing channel: CD4-PerCP (sliding paraboloid, 100px)
    Saved: processed_CD4-PerCP.tif
  Processing channel: CD45RA-PacBlue (sliding paraboloid, 100px)
    Saved: processed_CD45RA-P

model_type argument is not used in v4.0.1+. Ignoring this argument...
Resizing is depricated in v4.0.1+


  Saving original image: (1002, 1004), dtype: uint16, unique_values: 13433
✓ Visualization saved: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/2/cellpose_segmentation_visualization.png
Detecting channel files in: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/2/
Found 6 channel files:
  - Actin-FITC.tif
  - CCR7-AF594.tif
  - CD19CAR-AF647.tif
  - CD4-PerCP.tif
  - CD45RA-PacBlue.tif
  - cellpose_prob_map.tif

Preprocessing channels...
  Processing channel: Actin-FITC (sliding paraboloid, 100px)
    Saved: processed_Actin-FITC.tif
  Processing channel: CCR7-AF594 (sliding paraboloid, 100px)
    Saved: processed_CCR7-AF594.tif
  Processing channel: CD19CAR-AF647 (sliding paraboloid, 100px)
    Saved: processed_CD19CAR-AF647.tif
  Processing channel: CD4-PerCP (sliding paraboloid, 100px)
    Saved: processed_CD4-PerCP.tif
  Processing channel: CD45RA-PacBlue (sliding paraboloid, 100px)
    Saved: processed_CD45RA-P

...

Resizing is depricated in v4.0.1+


  Saving original image: (1002, 1004), dtype: uint16, unique_values: 8752
✓ Visualization saved: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/4/cellpose_segmentation_visualization.png
Detecting channel files in: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/4/
Found 6 channel files:
  - Actin-FITC.tif
  - CCR7-AF594.tif
  - CD19CAR-AF647.tif
  - CD4-PerCP.tif
  - CD45RA-PacBlue.tif
  - cellpose_prob_map.tif

Preprocessing channels...
  Processing channel: Actin-FITC (sliding paraboloid, 100px)
    Saved: processed_Actin-FITC.tif
  Processing channel: CCR7-AF594 (sliding paraboloid, 100px)
    Saved: processed_CCR7-AF594.tif
  Processing channel: CD19CAR-AF647 (sliding paraboloid, 100px)
    Saved: processed_CD19CAR-AF647.tif
  Processing channel: CD4-PerCP (sliding paraboloid, 100px)
    Saved: processed_CD4-PerCP.tif
  Processing channel: CD45RA-PacBlue (sliding paraboloid, 100px)
    Saved: processed_CD45RA-Pa

model_type argument is not used in v4.0.1+. Ignoring this argument...
Resizing is depricated in v4.0.1+


  Saving original image: (1002, 1004), dtype: uint16, unique_values: 11179
✓ Visualization saved: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/5/cellpose_segmentation_visualization.png
Detecting channel files in: /Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/sample1/5/
Found 6 channel files:
  - Actin-FITC.tif
  - CCR7-AF594.tif
  - CD19CAR-AF647.tif
  - CD4-PerCP.tif
  - CD45RA-PacBlue.tif
  - cellpose_prob_map.tif

Preprocessing channels...
  Processing channel: Actin-FITC (sliding paraboloid, 100px)
    Saved: processed_Actin-FITC.tif
  Processing channel: CCR7-AF594 (sliding paraboloid, 100px)
    Saved: processed_CCR7-AF594.tif
  Processing channel: CD19CAR-AF647 (sliding paraboloid, 100px)
    Saved: processed_CD19CAR-AF647.tif
  Processing channel: CD4-PerCP (sliding paraboloid, 100px)
    Saved: processed_CD4-PerCP.tif
  Processing channel: CD45RA-PacBlue (sliding paraboloid, 100px)
    Saved: processed_CD45RA-P

{'success': True,
 'error': None,
 'total_images': 5,
 'total_processed': 5,
 'total_failed': 0}

## Result Summary

In [7]:
if "donor_result" not in globals():
    raise RuntimeError("Run the 'Run Processing' cell first so donor_result is defined.")

summary = {
    "success": donor_result.success,
    "error": donor_result.error,
    "donor_dir": str(donor_result.donor_dir),
    "total_images": donor_result.total_images,
    "total_processed": donor_result.total_processed,
    "total_failed": donor_result.total_failed,
    "donor_table": str(donor_result.donor_table.output_path) if donor_result.donor_table and donor_result.donor_table.output_path else None,
}

summary

{'success': True,
 'error': None,
 'donor_dir': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241',
 'total_images': 5,
 'total_processed': 5,
 'total_failed': 0,
 'donor_table': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/all_samples_combined.csv'}

In [8]:
if "donor_result" not in globals():
    raise RuntimeError("Run the 'Run Processing' cell first so donor_result is defined.")

image_result_rows = []
for image_result in donor_result.image_results:
    image_result_rows.append({
        "sample": image_result.sample_name,
        "image": image_result.image_name,
        "success": image_result.success,
        "error": image_result.error,
        "result_keys": sorted(image_result.results.keys()),
    })

image_result_rows

[{'sample': 'sample1',
  'image': '1',
  'success': True,
  'error': None,
  'result_keys': ['combine',
   'imagej',
   'load_rois',
   'measurements',
   'padded_ccr7',
   'padded_cd45ra',
   'padded_cells',
   'preprocess_channels',
   'raw_actin',
   'raw_ccr7',
   'raw_cd45ra',
   'segmentation']},
 {'sample': 'sample1',
  'image': '2',
  'success': True,
  'error': None,
  'result_keys': ['combine',
   'imagej',
   'load_rois',
   'measurements',
   'padded_ccr7',
   'padded_cd45ra',
   'padded_cells',
   'preprocess_channels',
   'raw_actin',
   'raw_ccr7',
   'raw_cd45ra',
   'segmentation']},
 {'sample': 'sample1',
  'image': '3',
  'success': True,
  'error': None,
  'result_keys': ['combine',
   'imagej',
   'load_rois',
   'measurements',
   'padded_ccr7',
   'padded_cd45ra',
   'padded_cells',
   'preprocess_channels',
   'raw_actin',
   'raw_ccr7',
   'raw_cd45ra',
   'segmentation']},
 {'sample': 'sample1',
  'image': '4',
  'success': True,
  'error': None,
  'result_key

In [9]:
if "image_result_rows" not in globals():
    raise RuntimeError("Run the image result summary cell first so image_result_rows is defined.")

failed_images = [row for row in image_result_rows if not row["success"]]
failed_images

[]

## Rebuild Selected Donor CSV


In [10]:
# This rebuilds sample-level and donor-level CSVs using the same selected samples/images from config.
donor_table_result = build_donor_table(
    config,
    verbose=VERBOSE,
)

if not donor_table_result.success:
    raise RuntimeError(donor_table_result.error)

{
    "output_path": str(donor_table_result.output_path),
    "rows": donor_table_result.rows,
    "columns": donor_table_result.columns,
}


{'output_path': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/all_samples_combined.csv',
 'rows': 77,
 'columns': ('global_cell_id',
  'unique_id',
  'sample',
  'image',
  'cell_id',
  'cd4_median',
  'cd4_mean',
  'ccr7_median',
  'ccr7_mean',
  'cd45ra_median',
  'cd45ra_mean',
  'cd19car_median',
  'cd19car_mean',
  'actin_mean',
  'actin_median',
  'actin_std',
  'actin_min',
  'actin_max',
  'actin_intden',
  'actin_rawintden',
  'cd4_std',
  'cd4_min',
  'cd4_max',
  'cd4_intden',
  'cd4_rawintden',
  'cd45ra_std',
  'cd45ra_min',
  'cd45ra_max',
  'cd45ra_intden',
  'cd45ra_rawintden',
  'cd19car_std',
  'cd19car_min',
  'cd19car_max',
  'cd19car_intden',
  'cd19car_rawintden',
  'ccr7_std',
  'ccr7_min',
  'ccr7_max',
  'ccr7_intden',
  'ccr7_rawintden')}

## Plot Selected Intensity Distributions


In [11]:
if "donor_table_result" not in globals():
    raise RuntimeError("Run the 'Rebuild Selected Donor CSV' cell first.")

from dlbcl_pipeline.plotting.intensity_distributions import plot_donor_intensity_distributions

plot_result = plot_donor_intensity_distributions(
    donor_dir=config.donor_dir,
    csv_file=Path(donor_table_result.output_path).name,
    output_file="selected_images_intensity_histograms.png",
    donor_label=config.donor_dir.name,
    verbose=VERBOSE,
)

if not plot_result["success"]:
    raise RuntimeError(plot_result["error"])

plot_result


{'success': True,
 'figure_path': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/selected_images_intensity_histograms.png',
 'num_cells': 77,
 'stats': {'Actin-FITC': {'column': 'actin_mean',
   'n': 77,
   'mean': 3936.1395012987014,
   'std': 1557.2617901801991,
   'median': 3830.5264},
  'CD4-PerCP': {'column': 'cd4_mean',
   'n': 77,
   'mean': 373.61074675324676,
   'std': 93.82083621980834,
   'median': 365.9256},
  'CD45RA-PacBlue': {'column': 'cd45ra_mean',
   'n': 77,
   'mean': 108.346512987013,
   'std': 21.92165908408941,
   'median': 103.6654},
  'CD19CAR-AF647': {'column': 'cd19car_mean',
   'n': 77,
   'mean': 48.62284415584415,
   'std': 3.913696701850724,
   'median': 47.8174},
  'CCR7-AF594': {'column': 'ccr7_mean',
   'n': 77,
   'mean': 74.27742597402596,
   'std': 21.005291094723095,
   'median': 70.7955}}}

## Classify Selected Donor CSV


In [ ]:
if "donor_table_result" not in globals():
    raise RuntimeError("Run the 'Rebuild Selected Donor CSV' cell first.")

from dlbcl_pipeline.classification.classify_tcells import classify_tcells

# Edit these thresholds after inspecting the intensity distribution plot.
CLASSIFICATION_THRESHOLDS = {
    "cd4": 260.0,
    "cd45ra": 120.0,
    "ccr7": 114.0,
    "cd19car": 430.0,
}

classification_result = classify_tcells(
    base_path=config.donor_dir,
    input_csv=Path(donor_table_result.output_path).name,
    output_csv=config.export.classified_csv,
    thresholds=CLASSIFICATION_THRESHOLDS,
    verbose=VERBOSE,
)

if not classification_result["success"]:
    raise RuntimeError(classification_result.get("error", "Classification failed"))

classification_result


## Export Formatted Channel Images


In [13]:
if "classification_result" not in globals():
    raise RuntimeError("Run the 'Classify Selected Donor CSV' cell first.")
if not CLEAN_CELL_LIST_PATH.exists():
    raise FileNotFoundError(f"Clean cell list not found: {CLEAN_CELL_LIST_PATH}")

try:
    donor_folder_path = config.donor_dir.relative_to(PROJECT_DATA_ROOT)
except ValueError:
    donor_folder_path = config.donor_dir

formatted_export_result = build_formatted_channels(
    root=PROJECT_DATA_ROOT,
    donor_folder_path=donor_folder_path,
    clean_csv=CLEAN_CELL_LIST_PATH,
    channels=EXPORT_CHANNELS,
    samples_to_process=config.samples_to_process,
    images_to_process=config.images_to_process,
    classified_csv=config.export.classified_csv,
    clean_output=CLEAN_FORMATTED_OUTPUT,
    verbose=VERBOSE,
)

formatted_export_result


{'success': True,
 'donor_count': 1,
 'channel_results': [{'donor_id': '109241',
   'donor_root': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241',
   'channel': 'actin',
   'source_dirname': 'padded_cells',
   'output_dir': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/formatted_actin',
   'copied': 63,
   'missing': 2,
   'indexed_sources': 77,
   'duplicate_name_overwrites': 0,
   'skipped_existing': 0,
   'dry_run': False},
  {'donor_id': '109241',
   'donor_root': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241',
   'channel': 'ccr7',
   'source_dirname': 'padded_ccr7',
   'output_dir': '/Users/taeeonkong/Desktop/DL Project/non-responder/01-03-2026 DLBCL 109241/formatted_ccr7',
   'copied': 63,
   'missing': 2,
   'indexed_sources': 77,
   'duplicate_name_overwrites': 0,
   'skipped_existing': 0,
   'dry_run': False},
  {'donor_id': '109241',
   'donor_root': '/Users/taeeonkong/Desktop/DL Project

In [ ]:
if "formatted_export_result" not in globals():
    raise RuntimeError("Run the formatted channel export cell first.")

formatted_counts = {}
missing_formatted_sources = []
for result in formatted_export_result.get("channel_results", []):
    output_dir = Path(result["output_dir"])
    channel = result["channel"]
    formatted_counts[channel] = {
        "output_dir": str(output_dir),
        "files": len(list(output_dir.glob("*.tif"))) if output_dir.exists() else 0,
        "copied": result["copied"],
        "missing": result["missing"],
    }
    for item in result.get("missing_entries", []):
        missing_formatted_sources.append({
            "channel": channel,
            "sample": item["sample"],
            "image": item["image"],
            "cell": item["cell"],
            "expected_source_dir": item["expected_source_dir"],
            "target_name": item["target_name"],
        })

{
    "counts": formatted_counts,
    "missing_sources": missing_formatted_sources,
}


## Inspect Channel Crop Artifacts


In [15]:
if "donor_folder_structure" not in globals():
    raise RuntimeError("Run the 'Model Folder Structure' cell first so donor_folder_structure is defined.")

artifact_dirs = (
    "raw_actin",
    "raw_ccr7",
    "raw_cd45ra",
    "padded_cells",
    "padded_ccr7",
    "padded_cd45ra",
)

artifact_rows = []
for sample in donor_folder_structure.samples:
    for image in sample.images:
        row = {
            "sample": sample.name,
            "image": image.name,
        }
        for dirname in artifact_dirs:
            folder = image.path / dirname
            row[dirname] = len(list(folder.glob("*.tif"))) if folder.exists() else None
        artifact_rows.append(row)

artifact_rows


[{'sample': 'sample1',
  'image': '1',
  'raw_actin': 15,
  'raw_ccr7': 15,
  'raw_cd45ra': 15,
  'padded_cells': 15,
  'padded_ccr7': 15,
  'padded_cd45ra': 15},
 {'sample': 'sample1',
  'image': '2',
  'raw_actin': 21,
  'raw_ccr7': 21,
  'raw_cd45ra': 21,
  'padded_cells': 21,
  'padded_ccr7': 21,
  'padded_cd45ra': 21},
 {'sample': 'sample1',
  'image': '3',
  'raw_actin': 17,
  'raw_ccr7': 17,
  'raw_cd45ra': 17,
  'padded_cells': 17,
  'padded_ccr7': 17,
  'padded_cd45ra': 17},
 {'sample': 'sample1',
  'image': '4',
  'raw_actin': 11,
  'raw_ccr7': 11,
  'raw_cd45ra': 11,
  'padded_cells': 11,
  'padded_ccr7': 11,
  'padded_cd45ra': 11},
 {'sample': 'sample1',
  'image': '5',
  'raw_actin': 13,
  'raw_ccr7': 13,
  'raw_cd45ra': 13,
  'padded_cells': 13,
  'padded_ccr7': 13,
  'padded_cd45ra': 13}]